# Parte 2: Modelagem Preditiva

## *Preparação*: leitura da base de dados e mudanças dos tipos das variáveis

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# 1. Carregamento da base de dados
database = pd.read_csv("klike_challenge_dataset.csv")

# 2. Padronização dos tipos de variáveis (Dtypes)
database['has_subtitle'] = database['has_subtitle'].astype('boolean')
database['date'] = pd.to_datetime(database['date'])

cols_nominal = ['platform', 'category', 'objective', 'format']
database[cols_nominal] = database[cols_nominal].astype('category')

database['text_density'] = pd.Categorical(
    database['text_density'], 
    categories=['low', 'medium', 'high'], 
    ordered=True
)

database['target_audience_age'] = pd.Categorical(
    database['target_audience_age'], 
    categories=['18-24', '25-34', '35-44', '45+'], 
    ordered=True
)

print(f"Dataset carregado com sucesso: {database.shape[0]} linhas e {database.shape[1]} colunas.")



Dataset carregado com sucesso: 500 linhas e 26 colunas.


## Primeira Etapa: Feature Engineering

In [ ]:
import statsmodels.api as sm
# --- 1. FEATURE ENGINEERING ---
# Criamos a nova feature 'watch_ratio' (Proporção do vídeo que foi assistida, limitada em 100%)
df_model = database.copy()
df_model['watch_ratio'] = (df_model['avg_watch_time_s'] / df_model['video_duration_s']).clip(upper=1.0)

# --- 2. SELEÇÃO CRITERIOSA E PREPARAÇÃO DAS VARIÁVEIS ---
# Variáveis de Criativo + Contexto de Plataforma + Métricas-Chave de Vídeo
content_features = ["video_duration_s", "format", "has_subtitle", "has_cta", "has_hook", "has_face", "text_density", "music_voice_ratio"]
context_features = ["platform", "category", "objective", "target_audience_age", "is_retargeting"]

# Transformamos categorias em variáveis binárias (One-Hot Encoding / Dummies)
all_features = context_features + content_features
X_features = pd.get_dummies(df_model[all_features], drop_first=True, dtype=float)

# Preenchimento de eventuais nulos pela mediana (para não perder nenhuma linha)
X_features = X_features.fillna(X_features.median())

# Adicionamos a constante (intercepto da regressão)
X_with_const = sm.add_constant(X_features)
y_target = df_model['klike_score']

# --- 3. TREINAMENTO DO MODELO OLS & DIAGNÓSTICO ESTATÍSTICO ---
ols_model = sm.OLS(y_target, X_with_const).fit()

# Exibição da tabela clássica de Econometria / Estatística
print(ols_model.summary())


TypeError: Invalid value '1.0' for dtype 'boolean'